# Week 49, Day 6: Debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w049d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
def products_except_self(nums):
    out = [1] * len(nums)
    left = 1
    for i, x in enumerate(nums):
        out[i] = left            # the product of everything before i
        left *= x
    right = 1
    for i in range(len(nums) - 1, -1, -1):
        out[i] *= right          # times the product of everything after i
        right *= nums[i]
    return out

print(products_except_self([1, 2, 3, 4]), products_except_self([2, 0, 3]), products_except_self([0, 4, 0]))

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self._d = OrderedDict()            # least recently used first

    def get(self, key):
        if key not in self._d:
            return None
        self._d.move_to_end(key)
        return self._d[key]

    def put(self, key, value):
        self._d[key] = value
        self._d.move_to_end(key)
        if len(self._d) > self.capacity:
            self._d.popitem(last=False)

    def keys(self):
        return list(self._d)

c = LRUCache(2)
c.put("a", 1); c.put("b", 2); c.get("a"); c.put("c", 3)
print(c.keys(), c.get("b"))

In [ ]:
from collections import deque

def window_max(nums, k):
    out, dq = [], deque()                  # indices; their values decrease front to back
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()                       # they can never be a maximum again
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()                   # left the window
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out

print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

In [ ]:
from collections import deque

class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def save(root):
    out, queue = [], deque([root])
    while queue:
        n = queue.popleft()
        if n is None:
            out.append("#")
        else:
            out.append(str(n.v))
            queue.append(n.l)
            queue.append(n.r)
    while out and out[-1] == "#":
        out.pop()
    return ",".join(out)

def load(text):
    if not text:
        return None
    items = text.split(",")
    root = T(int(items[0]))
    queue, i = deque([root]), 1
    while queue and i < len(items):
        node = queue.popleft()
        for side in ("l", "r"):
            if i < len(items) and items[i] != "#":
                child = T(int(items[i]))
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

print(save(load("1,2,3,#,4")), repr(save(None)))

In [ ]:
def find_words(grid, words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    R, C = len(grid), len(grid[0]) if grid else 0
    found = set()

    def search(r, c, node, used):
        node = node.get(grid[r][c])
        if node is None:
            return                          # no stored word continues this way
        if "$" in node:
            found.add(node["$"])
        used.add((r, c))
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and (nr, nc) not in used:
                search(nr, nc, node, used)
        used.discard((r, c))                # backtrack

    for r in range(R):
        for c in range(C):
            search(r, c, root, set())
    return sorted(found)

print(find_words(["oaan", "etae", "ihkr", "iflv"], ["oath", "pea", "eat", "rain"]))

### Exercise: Second chance: running products

Write `max_product_subarray(nums)`: the largest product of a non-empty contiguous run, in O(n). Keep, at each position, both the largest and the smallest product of a run ending there (a negative number turns the smallest into the largest).

In [ ]:
def max_product_subarray(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, math\ndef test_products():\n    \"products\"\n    assert max_product_subarray([2, 3, -2, 4]) == 6 and max_product_subarray([-2, 0, -1]) == 0 and max_product_subarray([-2, 3, -4]) == 24\n    rng = random.Random(1)\n    for _ in range(300):\n        xs = [rng.randint(-3, 3) for _ in range(rng.randint(1, 10))]\n        assert max_product_subarray(xs) == max(math.prod(xs[i:j]) for i in range(len(xs)) for j in range(i + 1, len(xs) + 1))")

### Exercise: Second chance: a least-frequently-used cache

Write `LFUCache(capacity)` with `get(key)` and `put(key, value)` like B2, but evicting the key used **least often** (each get or put of a key counts as a use), and among those, the least recently used. Keep a dict from key to `(value, uses)` and, for each use count, an `OrderedDict` of keys in recency order, plus the current smallest use count, so every operation is O(1).

In [ ]:
from collections import OrderedDict, defaultdict

class LFUCache:
    def __init__(self, capacity):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_lfu():\n    \"least frequently used\"\n    c = LFUCache(2)\n    c.put(1, 1)\n    c.put(2, 2)\n    assert c.get(1) == 1\n    c.put(3, 3)\n    assert c.get(2) is None and c.get(3) == 3\n    c.put(4, 4)\n    assert c.get(1) is None and c.get(3) == 3 and c.get(4) == 4\n    z = LFUCache(0)\n    z.put(1, 1)\n    assert z.get(1) is None\ndef test_random():\n    \"against a slow version\"\n    rng = random.Random(2)\n    c = LFUCache(3)\n    vals, uses, last, t = {}, {}, {}, 0\n    for i in range(3000):\n        k = rng.randrange(6)\n        t += 1\n        if rng.random() < 0.5:\n            want = vals.get(k)\n            assert c.get(k) == want\n            if k in vals:\n                uses[k] += 1\n                last[k] = t\n        else:\n            if k not in vals and len(vals) == 3:\n                victim = min(vals, key=lambda x: (uses[x], last[x]))\n                for d in (vals, uses, last):\n                    del d[victim]\n            c.put(k, i)\n            vals[k] = i\n            uses[k] = uses.get(k, 0) + 1\n            last[k] = t")

### Exercise: Second chance: a window of bounded spread

Write `longest_calm(nums, limit)`: the length of the longest contiguous run whose maximum minus minimum is at most `limit`, in O(n) with a sliding window and two monotonic deques (one for the maximum, one for the minimum).

In [ ]:
from collections import deque

def longest_calm(nums, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_calm():\n    \"calm runs\"\n    assert longest_calm([8, 2, 4, 7], 4) == 2 and longest_calm([10, 1, 2, 4, 7, 2], 5) == 4 and longest_calm([], 3) == 0\n    rng = random.Random(3)\n    for _ in range(200):\n        xs = [rng.randint(0, 9) for _ in range(rng.randint(0, 15))]\n        lim = rng.randint(0, 5)\n        assert longest_calm(xs, lim) == max([j - i for i in range(len(xs)) for j in range(i + 1, len(xs) + 1) if max(xs[i:j]) - min(xs[i:j]) <= lim] + [0])")

### Exercise: Second chance: a tree from its levels

Using the same format as B4 (level order, `#` for missing children of real nodes, trailing `#`s removed), write `right_view(text)`: the value of the last node on each level of the saved tree, **without building the tree**: walk the items with a queue of "how many children slots does the next level have".

In [ ]:
def right_view(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_view():\n    \"right side\"\n    assert right_view(\"1,2,3,#,5,#,4\") == [1, 3, 4] and right_view(\"\") == [] and right_view(\"1,2\") == [1, 2]\n    assert right_view(\"1,2,3,4\") == [1, 3, 4] and right_view(\"1,#,2,#,3\") == [1, 2, 3]")

### Exercise: Second chance: replacing with dictionary words

Write `segment(text, words)`: split a string with no spaces into dictionary words, returning one list of words that joins back to `text` (any one; preferring, at each position, the **longest** word that still leads to a full split), or `None` if impossible. Use a trie to find every word starting at a position in one walk, and remember positions already known to fail in a set so no position is retried (otherwise this is exponential).

In [ ]:
def segment(text, words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef can(text, words):\n    ok = [True] + [False] * len(text)\n    for j in range(1, len(text) + 1):\n        ok[j] = any(ok[i] and text[i:j] in words for i in range(j))\n    return ok[-1]\ndef test_segment():\n    \"segmenting\"\n    assert segment(\"applepenapple\", [\"apple\", \"pen\"]) == [\"apple\", \"pen\", \"apple\"]\n    assert segment(\"catsandog\", [\"cats\", \"dog\", \"sand\", \"and\", \"cat\"]) is None and segment(\"\", [\"a\"]) == []\n    assert segment(\"pineapplepen\", [\"apple\", \"pen\", \"applepen\", \"pine\", \"pineapple\"]) == [\"pineapple\", \"pen\"]\n    assert segment(\"a\" * 40 + \"b\", [\"a\", \"aa\", \"aaa\"]) is None\n    rng = random.Random(4)\n    for _ in range(200):\n        words = list({\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 5))})\n        text = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 12)))\n        got = segment(text, words)\n        assert (got is not None) == can(text, set(words)) and (got is None or (\"\".join(got) == text and all(w in words for w in got)))")

### Bug hunt: Second chance: three more bugs

`kth_closest(points, k)` should return the k points nearest the origin, nearest first, ties to the earlier point, keeping a heap of at most k entries. It has three bugs: it keeps the k **farthest**, it compares squared distances computed wrongly (`x * 2 + y * 2`), and it returns the points farthest first. Fix them so this prints `[(1, 1), (-1, 1), (2, 0)]`.

In [ ]:
import heapq

def kth_closest(points, k):
    heap = []
    for i, (x, y) in enumerate(points):
        entry = (x * 2 + y * 2, i, (x, y))
        if len(heap) < k:
            heapq.heappush(heap, entry)
        elif entry > heap[0]:
            heapq.heapreplace(heap, entry)
    return [p for _, _, p in sorted(heap, reverse=True)]

print(kth_closest([(3, 3), (1, 1), (-1, 1), (5, -1), (2, 0), (0, 4)], 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"[(1, 1), (-1, 1), (2, 0)]\")\n    assert kth_closest([(2, 2), (1, 0), (0, 1)], 2) == [(1, 0), (0, 1)]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Your score

Write down your total for timed set 2 before reading on, and your set 1 total beside it. Then, as you go through each problem, note **why** you lost any marks:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Picked the wrong pattern**: you had the tools but reached for the wrong one. Note what should have pointed you to the right one.
3. **Edge case**: the main case worked, but empty input, a tie or a boundary failed.
4. **Ran out of time**: the plan, not the knowledge, was the problem.

| Score | What it means |
|---|---|
| 80 or more | Excellent: you can use Weeks 41 to 48 under pressure. |
| 60 to 79 | Good. Practise the patterns where you lost marks. |
| under 60 | Not yet. Spend extra time on the second chances below and the review days, then retry the set without looking at the solutions. |

## Part A

| Questions | The idea | Revise |
|---|---|---|
| 1 | amortised O(1) appends | [Week 41, Day 2](w041d2.html) |
| 2 | prefix sums: `prefix[j] - prefix[i]` is `nums[i:j]` | [Week 41, Day 5](w041d5.html) |
| 3 | Floyd's fast and slow pointers | [Week 42, Day 5](w042d5.html) |
| 4 | monotonic deques for window maxima | [Week 43, Day 3](w043d3.html) |
| 5 | tombstones in open addressing | [Week 44, Day 3](w044d3.html) |
| 6 | breadth-first order with a queue | [Week 45, Day 4](w045d4.html) |
| 7 | the four AVL cases | [Week 46, Day 3](w046d3.html) |
| 8 | k-way merging with a heap | [Week 47, Day 5](w047d5.html) |
| 9 | prefix hashes compare substrings in O(1) | [Week 48, Day 3](w048d3.html) |
| 10 | expanding around centres; the first longest wins on a tie | [Week 48, Day 4](w048d4.html) |

Question 10 catches people who don't trace carefully: "cdc" is as long as "aba", but the code only replaces `best` when a palindrome is **strictly** longer.

## Part B

Each model solution runs, so try it. The notes explain the key decision and the most common mistakes.

### B1: Products of the others

**The decision**: the answer at i is (product of the prefix before i) × (product of the suffix after i), the multiplicative version of prefix sums. **Common mistakes**: dividing the total product by `nums[i]`, which fails on zeros (and was forbidden); and an O(n²) double loop.

### B2: A least-recently-used cache

**The decision**: a dict for O(1) lookup plus an order you can change in O(1). `OrderedDict` is exactly that (internally, a dict and a doubly linked list, as in Week 42 Day 2). **Common mistakes**: forgetting that `put` on an existing key also makes it most recent; using a list for the order (`list.remove` is O(n)); and evicting before inserting, which evicts a key even when updating.

### B3: Window maximums

**The decision**: store **indices**, not values, so you can tell when the front has left the window. For the ranges, run the same code a second time with the comparison flipped for minimums. **Common mistakes**: storing values (then you can't tell which have expired); and taking `max` of each window, which is O(nk).

### B4: Saving a tree

**The decision**: only real nodes get children slots, so the loader's queue holds only real nodes (Week 46 Day 5). **Common mistakes**: queueing `None` children in the loader, which shifts every later value; recursion, which fails on the 10,000-node chain; and forgetting that `""` must load as an empty tree.

### B5: Words in a grid

**The decision**: search the grid **once**, guided by a trie of all the words, instead of once per word; a path is abandoned the moment no word has it as a prefix. Storing the whole word at its end node saves rebuilding it. Recursion is fine here: depth is at most the longest word. **Common mistakes**: not un-marking a cell after exploring from it (`used.discard`), which hides words that reuse that cell along another path; and reporting a word twice when it can be found in two places (a set fixes it).

### B6: Three bugs

The three fixes: lower-case the text before finding words (`text.lower()` with `[a-z]+`); replace the heap's top when the newcomer is **better**, `heap[0] < e`, not worse; and in `Entry.__lt__`, "worse" among equal counts means alphabetically **later**, so `self.word > other.word`. The third bug is the subtle one: whenever a heap orders by two keys in **different** directions (more uses better, earlier word better), write the comparison out and test a tie.

## Second chances

A fresh problem for each skill. Do them without looking back at the solutions above.

## Quiz

## What's next

**Your practice plan.** For each problem you lost marks on in either timed set, write one line: the problem, the kind of mistake, and the day you'll revisit. Then redo timed set 2's problems from scratch in a week's time without looking here: the real test of learning is whether it's still there later.

**Graphs.** After the rest day, Phase 4 continues with graphs: networks of things and the connections between them (roads, friendships, dependencies, web pages). You've already used their main ideas: breadth-first search on trees and word ladders, depth-first search with a stack, and heaps choosing what to explore next. Graphs drop the tree's rule that every node has one parent, which brings cycles, many paths and new questions: how to get anywhere, the shortest way, in what order to do dependent jobs.